# Data Mining Lab Assignment
## Data Lake Implementation using Apache Spark

**Name:** Koushik Yadav  
**Roll Number:** LCS2023028  

### Objective

The objective of this assignment is to implement a simple Data Lake
workflow using Apache Spark and the given sales dataset.

The workflow consists of:

- Storing the original dataset in the Bronze layer without modification.
- Inspecting the dataset using PySpark.
- Identifying duplicates, missing values, text inconsistencies,
  invalid numerical values, and invalid dates.
- Performing basic preprocessing using Spark.
- Storing the processed dataset in the Silver layer.
- Verifying the data before and after preprocessing.

No Gold layer or detailed data analysis is performed.

In [12]:
!pip install -q pyspark

In [13]:
import os
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [14]:
spark = SparkSession.builder \
    .appName("Koushik_Yadav_Data_Lake_Assignment") \
    .getOrCreate()

print("Spark started successfully.")
print("Spark version:", spark.version)

Spark started successfully.
Spark version: 4.0.4


In [15]:
bronze_dir = "/content/data_lake/bronze"
silver_dir = "/content/data_lake/silver/preprocessed_sales"

os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(silver_dir, exist_ok=True)

print("Data Lake structure created.")
print("Bronze:", bronze_dir)
print("Silver:", silver_dir)

Data Lake structure created.
Bronze: /content/data_lake/bronze
Silver: /content/data_lake/silver/preprocessed_sales


In [16]:
source_file = "/content/sales.csv"
bronze_file = os.path.join(bronze_dir, "sales.csv")

shutil.copy2(source_file, bronze_file)

print("Original sales.csv copied to Bronze layer.")
print("Bronze file:", bronze_file)

Original sales.csv copied to Bronze layer.
Bronze file: /content/data_lake/bronze/sales.csv


In [17]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_file)

print("Bronze dataset loaded successfully.")

Bronze dataset loaded successfully.


In [18]:
print("Bronze Dataset Schema")
print("=" * 40)

df.printSchema()

Bronze Dataset Schema
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [19]:
print("Sample records from Bronze layer")
print("=" * 40)

df.show(10, truncate=False)

Sample records from Bronze layer
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12

In [20]:
bronze_count = df.count()

print("Total records before preprocessing:", bronze_count)

Total records before preprocessing: 1000


In [21]:
duplicate_count = bronze_count - df.dropDuplicates().count()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 10


In [22]:
null_counts = df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in df.columns
])

print("Missing values in each column:")
null_counts.show(truncate=False)

Missing values in each column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [23]:
string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

print("Text columns:")
for column in string_columns:
    print("-", column)

Text columns:
- customer_id
- customer_name
- product
- category
- payment_method
- city
- state
- order_date
- order_status


In [24]:
for column in string_columns:
    print(f"\n--- {column} ---")
    df.select(column).distinct().show(30, truncate=False)


--- customer_id ---
+-----------+
|customer_id|
+-----------+
|C0114      |
|C0141      |
|C0243      |
|C0034      |
|C0154      |
|C0347      |
|C0130      |
|C0338      |
|C0167      |
|C0179      |
|C0195      |
|C0150      |
|C0033      |
|C0107      |
|C0016      |
|C0131      |
|C0191      |
|C0261      |
|C0057      |
|C0280      |
|C0046      |
|C0284      |
|C0102      |
|C0061      |
|C0337      |
|C0028      |
|C0058      |
|C0050      |
|C0139      |
|C0109      |
+-----------+
only showing top 30 rows

--- customer_name ---
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai      |
|Pooja Jain   

In [25]:
for column in string_columns:
    spaced_rows = df.filter(
        F.col(column).isNotNull() &
        (F.col(column) != F.trim(F.col(column)))
    ).count()

    print(f"{column}: {spaced_rows} rows with extra leading/trailing spaces")

customer_id: 0 rows with extra leading/trailing spaces
customer_name: 3 rows with extra leading/trailing spaces
product: 0 rows with extra leading/trailing spaces
category: 0 rows with extra leading/trailing spaces
payment_method: 0 rows with extra leading/trailing spaces
city: 3 rows with extra leading/trailing spaces
state: 0 rows with extra leading/trailing spaces
order_date: 0 rows with extra leading/trailing spaces
order_status: 0 rows with extra leading/trailing spaces


In [26]:
print("Quantity summary:")
df.select(
    F.min("quantity").alias("minimum"),
    F.max("quantity").alias("maximum"),
    F.avg("quantity").alias("average")
).show()

Quantity summary:
+-------+-------+-------+
|minimum|maximum|average|
+-------+-------+-------+
|     -2|     10|  5.355|
+-------+-------+-------+



In [27]:
invalid_quantity_count = df.filter(
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0)
).count()

print("Clearly invalid quantity records:", invalid_quantity_count)

Clearly invalid quantity records: 12


In [28]:
print("Unit price summary:")
df.select(
    F.min("unit_price").alias("minimum"),
    F.max("unit_price").alias("maximum"),
    F.avg("unit_price").alias("average")
).show()

Unit price summary:
+-------+-------+-------+
|minimum|maximum|average|
+-------+-------+-------+
|   -500|  81010| 5702.4|
+-------+-------+-------+



In [29]:
invalid_price_count = df.filter(
    F.col("unit_price").isNull() |
    (F.col("unit_price") <= 0)
).count()

print("Clearly invalid unit price records:", invalid_price_count)

Clearly invalid unit price records: 8


In [30]:
print("Discount percentage summary:")
df.select(
    F.min("discount_percent").alias("minimum"),
    F.max("discount_percent").alias("maximum"),
    F.avg("discount_percent").alias("average")
).show()

Discount percentage summary:
+-------+-------+-------+
|minimum|maximum|average|
+-------+-------+-------+
|     -5|    120| 19.948|
+-------+-------+-------+



In [31]:
invalid_discount_count = df.filter(
    F.col("discount_percent").isNull() |
    (F.col("discount_percent") < 0) |
    (F.col("discount_percent") > 100)
).count()

print("Clearly invalid discount records:", invalid_discount_count)

Clearly invalid discount records: 8


In [32]:
print("Sample order_date values:")
df.select("order_date").distinct().show(50, truncate=False)

Sample order_date values:
+----------+
|order_date|
+----------+
|2026-02-01|
|2025-06-03|
|2025-02-25|
|2026-02-27|
|2025-08-17|
|2026-07-23|
|2026-07-04|
|2025-09-14|
|2025-03-20|
|2025-04-05|
|2026-04-15|
|2026-06-05|
|2025-01-03|
|2025-12-07|
|2026-03-15|
|2025-12-30|
|2025-08-18|
|2026-09-11|
|2025-06-16|
|2026-09-06|
|2025-07-15|
|2026-08-24|
|2025/07/14|
|2025-05-06|
|2025-04-12|
|2026-02-15|
|2025-07-21|
|2025-05-23|
|2025-10-09|
|2026-01-05|
|2025-01-23|
|2026-01-28|
|2025-06-14|
|2026-02-25|
|2025-07-16|
|2025-01-28|
|2026-01-09|
|2025-03-12|
|2025-03-26|
|2025-12-14|
|2025-07-06|
|2025-07-08|
|2026-01-02|
|2025-05-09|
|2025-03-07|
|2025-06-01|
|2025-06-30|
|2025-07-29|
|2025-01-22|
|2026-01-13|
+----------+
only showing top 50 rows


In [34]:
date_check = df.withColumn(
    "parsed_date",
    F.coalesce(
        F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd')"),
        F.expr("try_to_timestamp(order_date, 'MM-dd-yyyy')"),
        F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")
    )
)

invalid_date_count = date_check.filter(
    F.col("order_date").isNotNull() &
    F.col("parsed_date").isNull()
).count()

print("Clearly invalid/unparseable date records:", invalid_date_count)

Clearly invalid/unparseable date records: 4


In [35]:
date_check.filter(
    F.col("order_date").isNotNull() &
    F.col("parsed_date").isNull()
).select(
    "order_date"
).distinct().show(truncate=False)

+----------+
|order_date|
+----------+
|31/02/2026|
|not-a-date|
|2025-02-30|
|2026-13-05|
+----------+



In [36]:
silver_df = df

In [37]:
silver_df = silver_df.dropDuplicates()

print("Rows after removing exact duplicates:", silver_df.count())

Rows after removing exact duplicates: 990


In [38]:
for column in string_columns:
    silver_df = silver_df.withColumn(
        column,
        F.trim(F.col(column))
    )

print("Extra leading/trailing spaces removed from text columns.")

Extra leading/trailing spaces removed from text columns.


In [39]:
for column in string_columns:
    silver_df = silver_df.withColumn(
        column,
        F.trim(F.col(column))
    )

print("Extra leading/trailing spaces removed from text columns.")

Extra leading/trailing spaces removed from text columns.


In [40]:
text_columns_to_standardize = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state"
]

for column in text_columns_to_standardize:
    silver_df = silver_df.withColumn(
        column,
        F.initcap(F.col(column))
    )

print("Obvious capitalization inconsistencies standardized.")

Obvious capitalization inconsistencies standardized.


In [41]:
silver_df = silver_df.fillna(
    {
        "customer_name": "Unknown",
        "product": "Unknown",
        "category": "Unknown",
        "payment_method": "Unknown",
        "city": "Unknown",
        "state": "Unknown"
    }
)

print("Missing text values handled.")

Missing text values handled.


In [42]:
valid_quantity_median = silver_df.filter(
    F.col("quantity") > 0
).approxQuantile(
    "quantity",
    [0.5],
    0.01
)[0]

print("Median valid quantity:", valid_quantity_median)

Median valid quantity: 5.0


In [43]:
silver_df = silver_df.withColumn(
    "quantity",
    F.when(
        (F.col("quantity").isNull()) | (F.col("quantity") <= 0),
        F.lit(valid_quantity_median)
    ).otherwise(F.col("quantity"))
)

print("Invalid and missing quantities handled.")

Invalid and missing quantities handled.


In [44]:
valid_price_median = silver_df.filter(
    F.col("unit_price") > 0
).approxQuantile(
    "unit_price",
    [0.5],
    0.01
)[0]

print("Median valid unit price:", valid_price_median)

Median valid unit price: 1810.0


In [45]:
silver_df = silver_df.withColumn(
    "unit_price",
    F.when(
        (F.col("unit_price").isNull()) | (F.col("unit_price") <= 0),
        F.lit(valid_price_median)
    ).otherwise(F.col("unit_price"))
)

print("Invalid and missing unit prices handled.")

Invalid and missing unit prices handled.


In [46]:
valid_discount_median = silver_df.filter(
    (F.col("discount_percent") >= 0) &
    (F.col("discount_percent") <= 100)
).approxQuantile(
    "discount_percent",
    [0.5],
    0.01
)[0]

print("Median valid discount percentage:", valid_discount_median)

Median valid discount percentage: 20.0


In [47]:
silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent").isNull()) |
        (F.col("discount_percent") < 0) |
        (F.col("discount_percent") > 100),
        F.lit(valid_discount_median)
    ).otherwise(F.col("discount_percent"))
)

print("Invalid and missing discount percentages handled.")

Invalid and missing discount percentages handled.


In [48]:
silver_df = silver_df.withColumn(
    "parsed_order_date",
    F.coalesce(
        F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd')"),
        F.expr("try_to_timestamp(order_date, 'MM-dd-yyyy')"),
        F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")
    )
)

silver_df = silver_df.withColumn(
    "order_date",
    F.when(
        F.col("parsed_order_date").isNotNull(),
        F.date_format(F.col("parsed_order_date"), "yyyy-MM-dd")
    ).otherwise(F.lit(None))
).drop("parsed_order_date")

print("Dates converted to the consistent YYYY-MM-DD format.")

Dates converted to the consistent YYYY-MM-DD format.


### Date Handling

Valid dates were converted to the consistent `YYYY-MM-DD` format.

Dates that could not be reliably parsed, including malformed or impossible
calendar dates, were converted to NULL rather than assigning an arbitrary
date. The corresponding sales records were preserved.

In [49]:
print("Remaining invalid quantities:",
      silver_df.filter(
          F.col("quantity").isNull() | (F.col("quantity") <= 0)
      ).count())

print("Remaining invalid unit prices:",
      silver_df.filter(
          F.col("unit_price").isNull() | (F.col("unit_price") <= 0)
      ).count())

print("Remaining invalid discounts:",
      silver_df.filter(
          F.col("discount_percent").isNull() |
          (F.col("discount_percent") < 0) |
          (F.col("discount_percent") > 100)
      ).count())

Remaining invalid quantities: 0
Remaining invalid unit prices: 0
Remaining invalid discounts: 0


In [50]:
remaining_invalid_dates = silver_df.filter(
    F.col("order_date").isNotNull() &
    F.to_date("order_date", "yyyy-MM-dd").isNull()
).count()

print("Remaining invalid non-null dates:", remaining_invalid_dates)

Remaining invalid non-null dates: 0


In [51]:
remaining_nulls = silver_df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in silver_df.columns
])

print("Remaining NULL values after preprocessing:")
remaining_nulls.show(truncate=False)

Remaining NULL values after preprocessing:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |0            |0      |0       |0       |0         |0               |0             |0   |0    |4         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [52]:
silver_count = silver_df.count()

print("Rows before preprocessing:", bronze_count)
print("Rows after preprocessing:", silver_count)
print("Rows removed:", bronze_count - silver_count)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed: 10


In [53]:
print("Sample records from the Silver DataFrame:")
silver_df.show(10, truncate=False)

Sample records from the Silver DataFrame:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripat

In [54]:
silver_df = silver_df.withColumn(
    "payment_method",
    F.when(F.lower(F.col("payment_method")) == "upi", "UPI")
     .when(F.lower(F.col("payment_method")) == "credit card", "Credit Card")
     .when(F.lower(F.col("payment_method")) == "debit card", "Debit Card")
     .when(F.lower(F.col("payment_method")) == "net banking", "Net Banking")
     .when(F.lower(F.col("payment_method")) == "cash on delivery", "Cash on Delivery")
     .otherwise(F.col("payment_method"))
)

print("Payment methods standardized.")

Payment methods standardized.


In [55]:
print("Standardized payment methods:")
silver_df.select("payment_method").distinct().orderBy("payment_method").show(
    truncate=False
)

Standardized payment methods:
+----------------+
|payment_method  |
+----------------+
|Cash on Delivery|
|Credit Card     |
|Debit Card      |
|Net Banking     |
|UPI             |
|Unknown         |
+----------------+



In [56]:
final_silver_count = silver_df.count()

print("Final Silver layer verification")
print("=" * 40)
print("Bronze records:", bronze_count)
print("Silver records:", final_silver_count)
print("Exact duplicate rows removed:", bronze_count - final_silver_count)

Final Silver layer verification
Bronze records: 1000
Silver records: 990
Exact duplicate rows removed: 10


In [57]:
silver_output_dir = "/content/data_lake/silver/preprocessed_sales"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_output_dir)

print("Silver data saved successfully.")
print("Location:", silver_output_dir)

Silver data saved successfully.
Location: /content/data_lake/silver/preprocessed_sales


In [58]:
print("Files created in Silver output directory:")

for file_name in os.listdir(silver_output_dir):
    print("-", file_name)

Files created in Silver output directory:
- .part-00000-28b792ce-03c8-48c5-a591-fe0718fc44d8-c000.csv.crc
- part-00000-28b792ce-03c8-48c5-a591-fe0718fc44d8-c000.csv
- ._SUCCESS.crc
- _SUCCESS


In [59]:
import glob
import shutil

part_files = glob.glob(
    os.path.join(silver_output_dir, "part-*.csv")
)

if not part_files:
    raise FileNotFoundError("Spark did not create a CSV part file.")

silver_csv = "/content/data_lake/silver/sales_silver.csv"

shutil.copy2(part_files[0], silver_csv)

print("Final Silver CSV created:")
print(silver_csv)

Final Silver CSV created:
/content/data_lake/silver/sales_silver.csv


In [60]:
silver_read_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(silver_csv)

print("Silver dataset read back successfully.")
print("Silver records:", silver_read_df.count())

Silver dataset read back successfully.
Silver records: 990


In [61]:
print("Sample records from final Silver dataset:")
silver_read_df.show(10, truncate=False)

Sample records from final Silver dataset:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripat

In [62]:
print("Final Silver schema:")
silver_read_df.printSchema()

Final Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)



## Preprocessing Summary

The following basic preprocessing operations were performed using Apache Spark:

1. Exact duplicate rows were identified and removed.
2. Missing values in text fields were handled using the value `Unknown`.
3. Leading and trailing spaces were removed from text fields.
4. Obvious capitalization inconsistencies in text fields were standardized.
5. Quantity values were validated, and clearly invalid or missing values were replaced using the median of valid quantities.
6. Unit price values were validated, and clearly invalid or missing values were replaced using the median of valid unit prices.
7. Discount percentages were validated against the range 0 to 100, and clearly invalid or missing values were replaced using the median of valid discounts.
8. Valid order dates were converted to the consistent `YYYY-MM-DD` format.
9. Clearly invalid or unparseable dates were converted to NULL while preserving their corresponding sales records.
10. The processed data was saved in the Silver layer and read back using Spark for verification.

The Bronze dataset was kept unchanged throughout the preprocessing workflow.

In [63]:
print("=" * 50)
print("FINAL DATA LAKE VERIFICATION")
print("=" * 50)

print("Bronze record count:", bronze_count)
print("Silver record count:", silver_read_df.count())
print("Silver CSV exists:", os.path.exists(silver_csv))

print("\nSilver sample:")
silver_read_df.show(5, truncate=False)

FINAL DATA LAKE VERIFICATION
Bronze record count: 1000
Silver record count: 990
Silver CSV exists: True

Silver sample:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
